# Task 5: SQL + Schema Design

Using SQLite in-memory to run SQL queries on Telco data

In [2]:
import pandas as pd
import sqlite3
import warnings; 
warnings.filterwarnings("ignore")

df = pd.read_csv("../data/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["MonthlyCharges"], inplace=True)

conn = sqlite3.connect(":memory:")
df.to_sql("customers", conn, index=False, if_exists="replace")
print("SQLite DB loaded with", len(df), "rows")
print("\nTable preview:")
pd.read_sql("SELECT * FROM customers LIMIT 3", conn)

SQLite DB loaded with 7043 rows

Table preview:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


## Query 1: Top 5 Customers by Revenue

In [3]:
q1 = """
SELECT 
    customerID,
    Contract,
    tenure,
    MonthlyCharges,
    TotalCharges,
    RANK() OVER (ORDER BY TotalCharges DESC) as revenue_rank
FROM customers
ORDER BY TotalCharges DESC
LIMIT 5
"""
result = pd.read_sql(q1, conn)
print("Top 5 Customers by Total Revenue:")
print(result.to_string(index=False))

Top 5 Customers by Total Revenue:
customerID Contract  tenure  MonthlyCharges  TotalCharges  revenue_rank
2889-FPWRM One year      72          117.80       8684.80             1
7569-NMZYQ Two year      72          118.75       8672.45             2
9739-JLPQJ Two year      72          117.50       8670.10             3
9788-HNGUT Two year      72          116.95       8594.40             4
8879-XUAHX Two year      71          116.25       8564.75             5


## Query 2: Monthly Revenue Growth

In [4]:
q2 = """
SELECT 
    CASE 
        WHEN tenure BETWEEN 1  AND 6  THEN '01-06 months'
        WHEN tenure BETWEEN 7  AND 12 THEN '07-12 months'
        WHEN tenure BETWEEN 13 AND 24 THEN '13-24 months'
        WHEN tenure BETWEEN 25 AND 48 THEN '25-48 months'
        ELSE '49+ months'
    END AS tenure_cohort,
    COUNT(*)                        AS customer_count,
    ROUND(AVG(MonthlyCharges), 2)   AS avg_monthly_charge,
    ROUND(SUM(TotalCharges), 2)     AS total_revenue,
    ROUND(SUM(TotalCharges)*100.0 / SUM(SUM(TotalCharges)) OVER(), 2) AS pct_of_total
FROM customers
GROUP BY tenure_cohort
ORDER BY tenure_cohort
"""
result = pd.read_sql(q2, conn)
print("Revenue by Tenure Cohort:")
print(result.to_string(index=False))

Revenue by Tenure Cohort:
tenure_cohort  customer_count  avg_monthly_charge  total_revenue  pct_of_total
 01-06 months            1470               54.84      211146.90          1.32
 07-12 months             705               58.95      390505.00          2.43
 13-24 months            1024               61.36     1153287.70          7.18
 25-48 months            1594               65.93     3810380.35         23.73
   49+ months            2250               73.79    10491304.35         65.34


## Query 3: Churn Rate Calculation

In [5]:
q3 = """
SELECT 
    Contract,
    InternetService,
    COUNT(*)                                        AS total_customers,
    SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END)   AS churned,
    ROUND(
        SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
    ) AS churn_rate_pct,
    ROUND(
        SUM(CASE WHEN Churn='Yes' THEN MonthlyCharges ELSE 0 END), 2
    ) AS monthly_revenue_at_risk
FROM customers
GROUP BY Contract, InternetService
ORDER BY churn_rate_pct DESC
"""
result = pd.read_sql(q3, conn)
print("Churn Rate by Contract & Internet Service:")
print(result.to_string(index=False))
print("\nMonth-to-month + Fiber Optic = highest churn risk!")

Churn Rate by Contract & Internet Service:
      Contract InternetService  total_customers  churned  churn_rate_pct  monthly_revenue_at_risk
Month-to-month     Fiber optic             2128     1162           54.61                100482.00
Month-to-month             DSL             1223      394           32.22                 18367.25
      One year     Fiber optic              539      104           19.29                 10571.95
Month-to-month              No              524       99           18.89                  1997.85
      One year             DSL              570       53            9.30                  3356.25
      Two year     Fiber optic              429       31            7.23                  3246.10
      One year              No              364        9            2.47                   190.25
      Two year             DSL              628       12            1.91                   805.70
      Two year              No              638        5            0.78   

## Bonus: Schema Design

In [6]:
schema = """
-- PROPOSED STAR SCHEMA for Telecom Analytics DWH
-- ────────────────────────────────────────────────────────

-- DIMENSION: dim_customer
CREATE TABLE dim_customer (
    customer_id     TEXT PRIMARY KEY,
    gender          TEXT,
    senior_citizen  INTEGER,    -- 0 or 1
    has_partner     TEXT,
    has_dependents  TEXT,
    signup_date     DATE
);

-- DIMENSION: dim_plan
CREATE TABLE dim_plan (
    plan_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    contract_type   TEXT,       -- Month-to-month, One year, Two year
    internet_svc    TEXT,       -- DSL, Fiber optic, No
    phone_svc       TEXT,
    streaming_tv    TEXT,
    streaming_movies TEXT
);

-- DIMENSION: dim_date
CREATE TABLE dim_date (
    date_id         INTEGER PRIMARY KEY,  -- YYYYMMDD
    year            INTEGER,
    month           INTEGER,
    quarter         INTEGER,
    month_name      TEXT
);

-- FACT: fact_billing (one row per customer per month)
CREATE TABLE fact_billing (
    billing_id          INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id         TEXT REFERENCES dim_customer(customer_id),
    plan_id             INTEGER REFERENCES dim_plan(plan_id),
    date_id             INTEGER REFERENCES dim_date(date_id),
    monthly_charges     REAL,
    total_charges       REAL,
    is_churned          INTEGER,    -- 0 or 1
    tenure_months       INTEGER,
    payment_method      TEXT
);

-- ────────────────────────────────────────────────────────
-- WHY STAR SCHEMA?
-- Fast aggregation queries (dashboards, reports)
-- Easy to extend (add new dimensions)
-- Optimized for BI tools (Tableau, Power BI, Metabase)
-- Churn rate = simple GROUP BY on fact table
"""
print(schema)


-- PROPOSED STAR SCHEMA for Telecom Analytics DWH
-- ────────────────────────────────────────────────────────

-- DIMENSION: dim_customer
CREATE TABLE dim_customer (
    customer_id     TEXT PRIMARY KEY,
    gender          TEXT,
    senior_citizen  INTEGER,    -- 0 or 1
    has_partner     TEXT,
    has_dependents  TEXT,
    signup_date     DATE
);

-- DIMENSION: dim_plan
CREATE TABLE dim_plan (
    plan_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    contract_type   TEXT,       -- Month-to-month, One year, Two year
    internet_svc    TEXT,       -- DSL, Fiber optic, No
    phone_svc       TEXT,
    streaming_tv    TEXT,
    streaming_movies TEXT
);

-- DIMENSION: dim_date
CREATE TABLE dim_date (
    date_id         INTEGER PRIMARY KEY,  -- YYYYMMDD
    year            INTEGER,
    month           INTEGER,
    quarter         INTEGER,
    month_name      TEXT
);

-- FACT: fact_billing (one row per customer per month)
CREATE TABLE fact_billing (
    billing_id          INTEGER PRIM